# Data Acquisition

This notebook handles fetching and consolidating raw intraday data for:
- NIFTY Futures
- NIFTY Options (CE & PE)

Source: NSE historical downloads (month-wise CSV iles)
to:
- data/raw/


In [3]:
import pandas as pd
import glob
import os

In [5]:
def load_and_concat_csv(folder_path):
    files = glob.glob(os.path.join(folder_path, "*.csv"))
    print(f"Found {len(files)} files in {folder_path}")

    df_list = []
    for file in files:
        df = pd.read_csv(file, low_memory=False)
        df_list.append(df)

    combined_df = pd.concat(df_list, ignore_index=True)
    return combined_df

In [6]:
futures = load_and_concat_csv("../data/raw/futures/")
options = load_and_concat_csv("../data/raw/options/")

futures.head(), options.head()

Found 12 files in ../data/raw/futures/
Found 10 files in ../data/raw/options/


(  Symbol         Date       Expiry      Open      High       Low     Close    \
 0    NIFTY  30-Apr-2025  26-Jun-2025  24482.50  24586.00  24453.50  24520.50   
 1    NIFTY  30-Apr-2025  29-May-2025  24369.90  24486.00  24348.70  24418.40   
 2    NIFTY  30-Apr-2025  31-Jul-2025  24617.60  24716.40  24590.00  24647.00   
 3    NIFTY  29-Apr-2025  31-Jul-2025  24689.00  24803.90  24636.00  24663.50   
 4    NIFTY  29-Apr-2025  26-Jun-2025  24586.70  24675.40  24506.00  24527.10   
 
       LTP   Settle Price   No. of contracts   Turnover * in   ₹ Lakhs  \
 0  24484.00       24520.50            3615.00                66494.63   
 1  24380.50       24418.40           69406.00              1271398.48   
 2  24618.00       24647.00             820.00                15166.49   
 3  24673.10       24663.50            1553.00                28789.33   
 4  24538.00       24527.10            5702.00               105106.06   
 
     Open Int   Change in OI   Underlying Value    
 0   1583625.0

In [8]:
def standardize_columns(df):
    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")
        .str.replace(".", "", regex=False)
        .str.replace("*", "", regex=False)
        .str.replace("₹", "rs", regex=False)
    )
    return df

futures = standardize_columns(futures)
options = standardize_columns(options)

futures.columns, options.columns

(Index(['symbol', 'date', 'expiry', 'open', 'high', 'low', 'close', 'ltp',
        'settle_price', 'no_of_contracts', 'turnover__in___rs_lakhs',
        'open_int', 'change_in_oi', 'underlying_value'],
       dtype='object'),
 Index(['symbol', 'date', 'expiry', 'option_type', 'strike_price', 'open',
        'high', 'low', 'close', 'ltp', 'settle_price', 'no_of_contracts',
        'turnover__in__rs_lakhs', 'premium_turnover__in___rs_lakhs', 'open_int',
        'change_in_oi', 'underlying_value'],
       dtype='object'))

In [9]:
options = options.rename(columns={
    "option_type": "option_type",
    "strike_price": "strike",
    "open_int": "open_interest",
    "no_of_contracts": "contracts"
})

futures = futures.rename(columns={
    "open_int": "open_interest",
    "no_of_contracts": "contracts"
})

In [10]:
options['timestamp'] = pd.to_datetime(options['date'])
futures['timestamp'] = pd.to_datetime(futures['date'])

In [11]:
options_ce = options[options['option_type'] == 'CE']
options_pe = options[options['option_type'] == 'PE']

options_ce.head(), options_pe.head()

(  symbol         date       expiry option_type   strike     open     high  \
 0  NIFTY  31-Oct-2025  04-Nov-2025          CE  23300.0        -        -   
 1  NIFTY  31-Oct-2025  29-Dec-2026          CE  26000.0  2550.00  2550.00   
 2  NIFTY  31-Oct-2025  30-Dec-2025          CE  23500.0  2650.00  2650.00   
 3  NIFTY  31-Oct-2025  31-Mar-2026          CE  19000.0        -        -   
 4  NIFTY  31-Oct-2025  24-Dec-2029          CE  29000.0        -        -   
 
        low    close      ltp settle_price contracts turnover__in__rs_lakhs  \
 0        -  2701.55  2700.40      2436.80         -                      -   
 1  2450.00  2458.00  2461.00      2458.00     51.00                1089.10   
 2  2590.00  2600.95  2602.75      2600.95     39.00                 763.59   
 3        -  5966.85        -      7168.75         -                      -   
 4        -  5305.80        -      4147.65         -                      -   
 
   premium_turnover__in___rs_lakhs open_interest chang

In [14]:
# Ensure CE & PE are real copies (fix SettingWithCopyWarning)
options_ce = options_ce.copy()
options_pe = options_pe.copy()
futures = futures.copy()

# Numeric columns per dataset
options_num_cols = [
    'open','high','low','close','ltp',
    'strike','open_interest','contracts','underlying_value'
]

futures_num_cols = [
    'open','high','low','close','ltp',
    'open_interest','contracts','underlying_value'
]

# Convert OPTIONS numeric columns safely
for col in options_num_cols:
    if col in options_ce.columns:
        options_ce.loc[:, col] = pd.to_numeric(options_ce[col], errors='coerce')
        options_pe.loc[:, col] = pd.to_numeric(options_pe[col], errors='coerce')

# Convert FUTURES numeric columns safely
for col in futures_num_cols:
    if col in futures.columns:
        futures.loc[:, col] = pd.to_numeric(futures[col], errors='coerce')

In [15]:
options_ce.dtypes, options_pe.dtypes, futures.dtypes

(symbol                                     object
 date                                       object
 expiry                                     object
 option_type                                object
 strike                                    float64
 open                                       object
 high                                       object
 low                                        object
 close                                      object
 ltp                                        object
 settle_price                               object
 contracts                                  object
 turnover__in__rs_lakhs                     object
 premium_turnover__in___rs_lakhs            object
 open_interest                              object
 change_in_oi                               object
 underlying_value                           object
 timestamp                          datetime64[ns]
 dtype: object,
 symbol                                     object
 date          

In [16]:
options_ce = options_ce.dropna(subset=['strike','close'])
options_pe = options_pe.dropna(subset=['strike','close'])
futures = futures.dropna(subset=['close'])

In [17]:
options_ce = options_ce.sort_values("timestamp")
options_pe = options_pe.sort_values("timestamp")
futures = futures.sort_values("timestamp")

In [18]:
futures.to_csv("../data/raw/nifty_futures_intraday.csv", index=False)
options_ce.to_csv("../data/raw/nifty_options_ce_intraday.csv", index=False)
options_pe.to_csv("../data/raw/nifty_options_pe_intraday.csv", index=False)

In [19]:
print(futures.shape, options_ce.shape, options_pe.shape)

(750, 15) (196543, 18) (200282, 18)
